In [1]:
# @title
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.8 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import cv2
import numpy as np


def crop_plate(image: np.ndarray, bbox_xyxy: tuple[int, int, int, int]) -> np.ndarray:
    x1, y1, x2, y2 = bbox_xyxy
    h, w = image.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    return image[y1:y2, x1:x2].copy()

In [3]:
import cv2
import os
from pathlib import Path
from ultralytics import YOLO

# 1. Setup Model
model = YOLO("/content/drive/MyDrive/AlgerPlate Project/runs/detect/yolov8s-ep50/weights/best.pt")
model.to("cpu")

# 2. Setup Paths
save_dir = "/content/drive/MyDrive/AlgerPlate Project/data/LP_data"
os.makedirs(save_dir, exist_ok=True)

# Combine all images into one flat list
# Note: use .glob or rglob to ensure we only grab image files
base_path = Path("/content/drive/MyDrive/AlgerPlate Project/data/images")
imgs = []
for folder in ["train", "val", "test"]:
    folder_path = base_path / folder
    if folder_path.exists():
        imgs.extend(list(folder_path.glob("*.jpg")) + list(folder_path.glob("*.png")))

print(f"Total images found: {len(imgs)}")

# 3. Processing Loop
for p in imgs:
    img = cv2.imread(str(p))
    if img is None:
        continue

    results = model(str(p), verbose=False)

    # Access the first result (one image)
    res = results[0]

    if len(res.boxes) == 0:
        print(f"No detection on {p.name}")
        continue

    # LOOP THROUGH ALL BOXES (Challenge fix)
    for i, box in enumerate(res.boxes):
        # Extract coordinates and confidence
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])

        # Crop the plate using your crop_plate function
        # Ensure crop_plate is defined in your notebook!
        try:
            cropped_plate = crop_plate(img, (x1, y1, x2, y2))

            # Create a unique filename: OriginalName_BoxIndex.png
            save_name = f"{p.stem}_box{i}.png"
            save_path = os.path.join(save_dir, save_name)

            # Save to Google Drive (Challenge fix)
            success = cv2.imwrite(save_path, cropped_plate)

            if success:
                print(f"Saved: {save_name} (Conf: {conf:.2f})")
            else:
                print(f"Failed to write image to {save_path}")

        except Exception as e:
            print(f"Error cropping box {i} in {p.name}: {e}")

print("Dataset building complete.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Total images found: 1950
Saved: ce549cb2-img_0620_box0.png (Conf: 0.93)
Saved: 37eed58e-img_1052_box0.png (Conf: 0.91)
Saved: 34ebe64f-img_1280_box0.png (Conf: 0.88)
Saved: 34ebe64f-img_1280_box1.png (Conf: 0.87)
Saved: 41a3eb61-img_1197_box0.png (Conf: 0.89)
Saved: cbb2c3e1-img_1130_box0.png (Conf: 0.90)
Saved: 20384117-img_1136_box0.png (Conf: 0.88)
Saved: f6015c80-img_1317_box0.png (Conf: 0.90)
Saved: 1ce387f0-img_0898_box0.png (Conf: 0.89)
Saved: 932b0787-img_1365_box0.png (Conf: 0.91)
Saved: 767e5ebd-img_1352_box0.png (Conf: 0.93)
Saved: bd38376e-img_0816_box0.png (Conf: 0.88)
Saved: 4fdcf2ed-img_0986_box0.png (Conf: 0.94)
Saved: f452b478-img_0735_box0.png (Conf: 0.90)
Saved:

In [5]:
print(f"Total images created: {len(list(Path(save_dir).iterdir()))}")

Total images created: 2143
